# ANPR YOLO Training Walkthrough

This notebook is for understanding the project step by step.

Goal:

1. Understand what YOLO does in this ANPR project.
2. Check the Kaggle dataset folder.
3. Convert annotations into YOLO format.
4. Train YOLO using transfer learning.
5. Validate the trained model.
6. Test prediction on a few images.
7. Know what to copy into the Streamlit app later.

Important: YOLO only detects the number plate box. OCR reads the text later.

## Step 0: Project Idea In Simple Words

ANPR means Automatic Number Plate Recognition.

The pipeline is:

```text
Vehicle image
    ↓
YOLO detects plate location
    ↓
OpenCV crops/cleans the plate image
    ↓
EasyOCR reads the text
    ↓
Python rules validate Indian plate format
```

In this notebook we train only the YOLO part.

That means we are improving:

```text
Can the model find the number plate box?
```

We are not training OCR in this notebook.

In [3]:
from pathlib import Path

PROJECT_DIR = Path.cwd()
RAW_DATASET = PROJECT_DIR / "datasets" / "raw_indian_vehicle"
YOLO_DATASET = PROJECT_DIR / "datasets" / "plate_yolo"

print("Project folder:", PROJECT_DIR)
print("Raw Kaggle dataset:", RAW_DATASET)
print("Prepared YOLO dataset:", YOLO_DATASET)
print("Raw dataset exists:", RAW_DATASET.exists())

Project folder: /Users/somtomar/Documents/anpr_project/License-Plate-Recognition-app
Raw Kaggle dataset: /Users/somtomar/Documents/anpr_project/License-Plate-Recognition-app/datasets/raw_indian_vehicle
Prepared YOLO dataset: /Users/somtomar/Documents/anpr_project/License-Plate-Recognition-app/datasets/plate_yolo
Raw dataset exists: True


## Step 1: Inspect The Kaggle Dataset

Before training, always inspect what you downloaded.

We need:

- images: `.jpg`, `.jpeg`, `.png`
- annotations: usually `.xml` for Pascal VOC or `.txt` for YOLO

Your downloaded dataset already appears to have XML annotations, which is good.

In [8]:
IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

images = [p for p in RAW_DATASET.rglob("*") if p.suffix.lower() in IMAGE_EXTS]
xml_files = list(RAW_DATASET.rglob("*.xml"))
txt_files = list(RAW_DATASET.rglob("*.txt"))

print("Images found:", len(images))
print("XML annotations found:", len(xml_files))
print("TXT files found:", len(txt_files))
print("First 5 images:")
for p in images[:5]:
    print("-", p.relative_to(PROJECT_DIR))

Images found: 1698
XML annotations found: 1697
TXT files found: 1
First 5 images:
- datasets/raw_indian_vehicle/google_images/car-wbs-MH20EE0943_00000.jpeg
- datasets/raw_indian_vehicle/google_images/c02b3989-8334-4cf7-b69c-bcd177e209d4___mqdefault.jpg.jpeg
- datasets/raw_indian_vehicle/google_images/car-wbs-HR26CT6702_00000.jpeg
- datasets/raw_indian_vehicle/google_images/car-wbs-TN45BA1065_00000.jpeg
- datasets/raw_indian_vehicle/google_images/f7ba7a93-e8d7-4253-b327-d87e3f2bb747___3e7fd381-0ae5-4421-8a70-279ee0ec1c61_2013-Nissan-Terrano-Rear.jpg


## Step 2: Convert Dataset To YOLO Format

YOLO expects labels like this:

```text
class_id x_center y_center width height
```

All values except `class_id` are normalized between 0 and 1.

Our project has a script that converts XML bounding boxes into YOLO labels.

Output structure:

```text
datasets/plate_yolo/
  images/train/
  images/val/
  labels/train/
  labels/val/
  dataset.yaml
```

In [11]:
!python3 scripts/prepare_yolo_dataset.py datasets/raw_indian_vehicle --out datasets/plate_yolo

YOLO dataset prepared
Raw images found: 1698
Annotated samples used: 1698
Skipped without annotations: 0
Train samples: 1359
Validation samples: 339
Dataset YAML: datasets/plate_yolo/dataset.yaml


## Step 3: Verify Prepared Dataset

Before training, check that train images and train labels count match.

If image count and label count are different, training may fail or behave badly.

In [16]:
train_images = list((YOLO_DATASET / "images" / "train").glob("*"))
val_images = list((YOLO_DATASET / "images" / "val").glob("*"))
train_labels = list((YOLO_DATASET / "labels" / "train").glob("*.txt"))
val_labels = list((YOLO_DATASET / "labels" / "val").glob("*.txt"))

print("Train images:", len(train_images))
print("Train labels:", len(train_labels))
print("Val images:", len(val_images))
print("Val labels:", len(val_labels))
print("Dataset YAML:")
print((YOLO_DATASET / "dataset.yaml").read_text())

Train images: 1359
Train labels: 1359
Val images: 339
Val labels: 339
Dataset YAML:
path: /Users/somtomar/Documents/anpr_project/License-Plate-Recognition-app/datasets/plate_yolo
train: images/train
val: images/val
names:
  0: plate



## Step 4: View One YOLO Label

This lets you understand what the training label looks like.

Example:

```text
0 0.512 0.731 0.210 0.055
```

Meaning:

- `0` = class id for plate
- `0.512` = x center of box
- `0.731` = y center of box
- `0.210` = box width
- `0.055` = box height

In [19]:
example_label = train_labels[0]
print("Example label file:", example_label.relative_to(PROJECT_DIR))
print(example_label.read_text())

Example label file: datasets/plate_yolo/labels/train/000618_video3_2190.txt
0 0.497934 0.850467 0.297521 0.071651



## Step 5: Transfer Learning Explanation

We will not train YOLO from zero.

We start from:

```text
yolo11n.pt
```

That model already knows general visual features like edges, corners, shapes, and object boundaries.

Then we fine-tune it on one class:

```text
plate
```

This is called transfer learning.

## Step 6: Train YOLO - Small Test Run

Run 10 epochs first.

Why only 10?

Because this checks whether:

- dataset is valid
- YOLO can train
- your laptop can handle it
- output files are created

If this works, later run 30 epochs.

In [ ]:
!python3 scripts/train_yolo_plate.py --data datasets/plate_yolo/dataset.yaml --epochs 10 --model yolo11n.pt --batch 8

New https://pypi.org/project/ultralytics/8.4.45 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.37 🚀 Python-3.10.19 torch-2.5.1 CPU (Apple M1)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=datasets/plate_yolo/dataset.yaml, degrees=0.0, deterministic=True, device=cpu, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=10, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=plate_train, nbs=64, 

## Step 7: Check Training Output

After training, YOLO saves files under:

```text
runs/detect/plate_train/
```

Most important file:

```text
runs/detect/plate_train/weights/best.pt
```

That is your trained detector.

In [ ]:
TRAIN_RUN = PROJECT_DIR / "runs" / "detect" / "plate_train"
BEST_MODEL = TRAIN_RUN / "weights" / "best.pt"

print("Training run exists:", TRAIN_RUN.exists())
print("Best model exists:", BEST_MODEL.exists())
print("Best model path:", BEST_MODEL)

if TRAIN_RUN.exists():
    print("
Files in training run:")
    for p in sorted(TRAIN_RUN.iterdir()):
        print("-", p.name)

## Step 8: Validate The Trained Model

Validation checks detection performance on the validation images.

YOLO reports metrics like precision, recall, and mAP.

Interview meaning:

- precision: when model detects a plate, how often is it correct?
- recall: how many real plates did it find?
- mAP: object detection quality metric based on box overlap and confidence

In [ ]:
from ultralytics import YOLO

if BEST_MODEL.exists():
    model = YOLO(str(BEST_MODEL))
    metrics = model.val(data=str(YOLO_DATASET / "dataset.yaml"), imgsz=640)
    print(metrics)
else:
    print("Train first. best.pt not found yet.")

## Step 9: Run Prediction On Sample Validation Images

This creates prediction images in:

```text
runs/detect/predict/
```

Open those images to visually check if boxes are correct.

In [ ]:
if BEST_MODEL.exists():
    sample_images = val_images[:8]
    model = YOLO(str(BEST_MODEL))
    results = model.predict([str(p) for p in sample_images], conf=0.25, save=True)
    print("Predicted", len(results), "images")
else:
    print("Train first. best.pt not found yet.")

## Step 10: Use The Trained Model In The App

Only do this after training works and predictions look decent.

Backup old model:

```bash
cp best.pt best_old.pt
```

Replace app model:

```bash
cp runs/detect/plate_train/weights/best.pt best.pt
```

Then run:

```bash
streamlit run app.py
```

Your Streamlit app loads `best.pt`, so it will now use your trained detector.

## Step 11: What To Say In Interview

Simple answer:

> I used transfer learning to fine-tune a pretrained YOLO model for Indian number plate detection. YOLO detects the plate box. Then the app crops that region, applies OpenCV preprocessing, uses EasyOCR to read the text, and validates the result using Indian plate-format rules.

If asked why OCR is separate:

> YOLO is an object detector. It finds where the plate is. OCR is needed to read the characters inside the plate.

If asked what training improved:

> Training improves plate localization, especially cases where the old detector missed the plate. OCR errors are handled separately with preprocessing and post-processing.